# 🍩 DeepSpringfield — Notebook d'Entraînement YOLOv8
## Détection de personnages des Simpsons

**Stratégie expérimentale :**
- Comparaison de **2 architectures** : `yolov8n`, `yolov8s`
- Pour chaque architecture, **2 configurations d'hyperparamètres** sont testées (lr + augmentation)
- Sélection automatique du meilleur modèle sur la validation (mAP@50)
- Le meilleur modèle est sauvegardé sur Google Drive pour l'évaluation finale

## 1. Installation & Imports

In [ ]:
!pip install ultralytics --quiet

import os
import glob
import yaml
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
from ultralytics import YOLO
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
else:
    print("Utilisation du CPU")

## 2. Montage Google Drive & Chemins

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = Path("/content/drive/MyDrive/M2 SISE/DL/DeepSpringfield_ComputerVision/dataset")
DATA_YAML    = DATASET_ROOT / "data.yaml"
TRAIN_IMG    = DATASET_ROOT / "train" / "images"
TRAIN_LBL    = DATASET_ROOT / "train" / "labels"
VALID_IMG    = DATASET_ROOT / "valid"  / "images"
VALID_LBL    = DATASET_ROOT / "valid"  / "labels"
TEST_IMG     = DATASET_ROOT / "test"   / "images"
TEST_LBL     = DATASET_ROOT / "test"   / "labels"

MODELS_DIR = Path("/content/drive/MyDrive/M2 SISE/DL/DeepSpringfield_ComputerVision") / "models"
MODELS_DIR.mkdir(exist_ok=True)

for p in [DATA_YAML, TRAIN_IMG, VALID_IMG, TEST_IMG]:
    print(f"{'[OK]' if p.exists() else '[MANQUANT]'} {p}")

## 3. Exploration du Dataset (EDA)

In [ ]:
with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config["names"]
NUM_CLASSES = config["nc"]

print("Configuration du dataset :")
print(f"  Nombre de classes : {NUM_CLASSES}")
print(f"  Classes           : {CLASS_NAMES}")

splits = {
    "Train"      : list(TRAIN_IMG.glob("*.jpg")) + list(TRAIN_IMG.glob("*.jpeg")) + list(TRAIN_IMG.glob("*.png")),
    "Validation" : list(VALID_IMG.glob("*.jpg"))  + list(VALID_IMG.glob("*.jpeg"))  + list(VALID_IMG.glob("*.png")),
    "Test"       : list(TEST_IMG.glob("*.jpg"))   + list(TEST_IMG.glob("*.jpeg"))   + list(TEST_IMG.glob("*.png")),
}

total = 0
print("\nNombre d'images par split :")
for split_name, imgs in splits.items():
    print(f"  {split_name:12s} : {len(imgs)} images")
    total += len(imgs)
print(f"  {'Total':12s} : {total} images")

In [ ]:
def parse_labels(label_dir):
    annotations = []
    for lbl_file in Path(label_dir).glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:])
                    annotations.append({"class_id": cls_id, "cx": cx, "cy": cy, "w": w, "h": h, "area": w * h})
    return pd.DataFrame(annotations)

df_train = parse_labels(TRAIN_LBL); df_train["split"] = "Train"
df_valid = parse_labels(VALID_LBL); df_valid["split"] = "Validation"
df_test  = parse_labels(TEST_LBL);  df_test["split"]  = "Test"
df_all   = pd.concat([df_train, df_valid, df_test], ignore_index=True)
df_all["class_name"] = df_all["class_id"].map(lambda i: CLASS_NAMES[i])

print(f"Total annotations : {len(df_all)}")
print(df_all.groupby(["split", "class_name"]).size().unstack(fill_value=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df_all["class_name"].value_counts().reindex(CLASS_NAMES)
colors = sns.color_palette("Set2", NUM_CLASSES)
axes[0].bar(CLASS_NAMES, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Distribution des classes (toutes données)", fontsize=13)
axes[0].set_xlabel("Personnage"); axes[0].set_ylabel("Nombre d'annotations")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

split_class = df_all.groupby(["class_name", "split"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
split_class.plot(kind="bar", ax=axes[1], color=["#2196F3", "#FF9800", "#4CAF50"], edgecolor="black")
axes[1].set_title("Distribution des classes par split", fontsize=13)
axes[1].set_xlabel("Personnage"); axes[1].set_ylabel("Nombre d'annotations")
axes[1].legend(title="Split"); axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("distribution_classes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_all["w"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution largeur (w)"); axes[0].set_xlabel("Largeur normalisée")

axes[1].hist(df_all["h"], bins=30, color="coral", edgecolor="white")
axes[1].set_title("Distribution hauteur (h)"); axes[1].set_xlabel("Hauteur normalisée")

axes[2].scatter(df_all["w"], df_all["h"], alpha=0.4, c=df_all["class_id"], cmap="tab10", s=20)
axes[2].set_title("Largeur vs Hauteur des bbox")
axes[2].set_xlabel("Largeur normalisée"); axes[2].set_ylabel("Hauteur normalisée")
for i, name in enumerate(CLASS_NAMES):
    axes[2].scatter([], [], label=name, c=[plt.cm.tab10(i / 10)])
axes[2].legend(fontsize=8)

plt.suptitle("Analyse des boîtes englobantes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("bbox_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def show_annotated_images(img_dir, lbl_dir, class_names, n=6, seed=42):
    random.seed(seed)
    img_files = list(Path(img_dir).glob("*.jpg")) + list(Path(img_dir).glob("*.jpeg"))
    img_files = [f for f in img_files if (Path(lbl_dir) / (f.stem + ".txt")).exists()]
    selected  = random.sample(img_files, min(n, len(img_files)))
    cols = 3; rows = (len(selected) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes = axes.flatten() if rows*cols > 1 else [axes]
    cmap = plt.cm.get_cmap("tab10")
    for ax, img_path in zip(axes, selected):
        img = Image.open(img_path).convert("RGB"); w_img, h_img = img.size
        ax.imshow(img)
        with open(Path(lbl_dir) / (img_path.stem + ".txt")) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = (cx - bw/2)*w_img; y1 = (cy - bh/2)*h_img
                ax.add_patch(patches.Rectangle((x1,y1), bw*w_img, bh*h_img,
                    linewidth=2, edgecolor=cmap(cls_id/10), facecolor="none"))
                ax.text(x1, y1-4, class_names[cls_id], color="white", fontsize=9,
                        fontweight="bold", bbox=dict(facecolor=cmap(cls_id/10), alpha=0.7, pad=1))
        ax.set_title(img_path.name[:30], fontsize=8); ax.axis("off")
    for ax in axes[len(selected):]: ax.axis("off")
    plt.suptitle("Exemples d'images annotées (Train)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("sample_annotations.png", dpi=150, bbox_inches="tight")
    plt.show()

show_annotated_images(TRAIN_IMG, TRAIN_LBL, CLASS_NAMES, n=6)

## 4. Stratégie Expérimentale

### Pourquoi comparer plusieurs modèles et configurations ?

Sur un petit dataset (≈150 images), le choix du modèle et des hyperparamètres a un impact majeur :
- Un modèle trop grand peut sur-apprendre (overfitting), c'est pourquoi on se limite à **yolov8n** et **yolov8s**
- Un modèle trop petit (**yolov8n**) peut manquer de capacité
- Le learning rate et l'augmentation influencent fortement la convergence

### Grille d'expériences

| Run | Modèle | LR initial | Augmentation | Optimizer |
|-----|--------|-----------|--------------|-----------|
| A | yolov8n | 1e-3 | Standard | AdamW |
| B | yolov8n | 5e-4 | Renforcée | AdamW |
| C | yolov8s | 1e-3 | Standard | AdamW |
| D | yolov8s | 5e-4 | Renforcée | AdamW |

**Augmentation standard** : paramètres YOLOv8 par défaut légèrement ajustés
**Augmentation renforcée** : mosaic=1.0, mixup=0.15, degrees=15, flipud=0.2, copy_paste=0.1

## 5. Définition des Configurations

In [ ]:
DEVICE     = 0 if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 640
BATCH_SIZE = 16
EPOCHS     = 80
PATIENCE   = 15
PROJECT    = "simpsons_experiments"

DATA_YAML_STR = str(DATA_YAML.resolve())

COMMON = dict(
    data          = DATA_YAML_STR,
    imgsz         = IMG_SIZE,
    batch         = BATCH_SIZE,
    epochs        = EPOCHS,
    patience      = PATIENCE,
    device        = DEVICE,
    project       = PROJECT,
    save          = True,
    plots         = True,
    verbose       = False,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    lrf           = 0.01,
)

AUG_STANDARD = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.05, copy_paste=0.0,
)

AUG_STRONG = dict(
    hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
    degrees=15.0, translate=0.15, scale=0.6,
    fliplr=0.5, flipud=0.2,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
)

EXPERIMENTS = [
    {"name": "run_A_nano_lr1e3_std",     "model": "yolov8n.pt", "lr0": 1e-3, "optimizer": "AdamW", "aug": AUG_STANDARD},
    {"name": "run_B_nano_lr5e4_strong",  "model": "yolov8n.pt", "lr0": 5e-4, "optimizer": "AdamW", "aug": AUG_STRONG},
    {"name": "run_C_small_lr1e3_std",    "model": "yolov8s.pt", "lr0": 1e-3, "optimizer": "AdamW", "aug": AUG_STANDARD},
    {"name": "run_D_small_lr5e4_strong", "model": "yolov8s.pt", "lr0": 5e-4, "optimizer": "AdamW", "aug": AUG_STRONG},
]

print(f"Nombre d'expériences : {len(EXPERIMENTS)}")
print(f"Device               : {DEVICE}")
print(f"Epochs max           : {EPOCHS} (early stopping patience={PATIENCE})")
print("\nRuns planifiés :")
for i, exp in enumerate(EXPERIMENTS):
    print(f"  [{i+1}] {exp['name']:<35} | modèle={exp['model']:<12} | lr={exp['lr0']} | optim={exp['optimizer']}")

## 6. Lancement des Expériences

In [ ]:
results_log = []

for i, exp in enumerate(EXPERIMENTS):
    print("\n" + "="*65)
    print(f"[{i+1}/{len(EXPERIMENTS)}] {exp['name']}")
    print("="*65)

    model = YOLO(exp["model"])

    train_kwargs = {
        **COMMON,
        **exp["aug"],
        "name"      : exp["name"],
        "lr0"       : exp["lr0"],
        "optimizer" : exp["optimizer"],
    }

    model.train(**train_kwargs)

    # Récupérer le best.pt — on teste tous les chemins possibles
    possible_paths = [
        Path("/content/runs/detect") / PROJECT / exp["name"] / "weights" / "best.pt",
        Path("/content/runs/detect") / exp["name"] / "weights" / "best.pt",
        Path(PROJECT) / exp["name"] / "weights" / "best.pt",
        Path("runs/detect") / exp["name"] / "weights" / "best.pt",
    ]

    best_pt = None
    for p in possible_paths:
        if p.exists():
            best_pt = p
            break

    if best_pt is None:
        print(f"❌ best.pt introuvable pour {exp['name']}, chemins testés :")
        for p in possible_paths:
            print(f"   {p}")
        continue

    print(f"✅ Modèle trouvé : {best_pt}")

    # Évaluation sur la validation
    best_model = YOLO(str(best_pt))
    metrics_val = best_model.val(
        data    = DATA_YAML_STR,
        split   = "val",
        imgsz   = IMG_SIZE,
        device  = DEVICE,
        verbose = False,
    )

    row = {
        "run"          : exp["name"],
        "model"        : exp["model"],
        "lr0"          : exp["lr0"],
        "optimizer"    : exp["optimizer"],
        "augmentation" : "renforcée" if exp["aug"] == AUG_STRONG else "standard",
        "mAP50_val"    : metrics_val.box.map50,
        "mAP50_95_val" : metrics_val.box.map,
        "precision_val": metrics_val.box.mp,
        "recall_val"   : metrics_val.box.mr,
        "best_pt"      : str(best_pt),
    }
    results_log.append(row)
    print(f"  → mAP@50 val : {row['mAP50_val']:.4f}  |  mAP@50-95 : {row['mAP50_95_val']:.4f}")

print("\n✅ Toutes les expériences sont terminées !")

## 7. Comparaison des Expériences

In [ ]:
df_results = pd.DataFrame(results_log)
df_results = df_results.sort_values("mAP50_val", ascending=False).reset_index(drop=True)

print("Classement des runs (trié par mAP@50 sur validation) :\n")
cols_display = ["run", "model", "lr0", "optimizer", "augmentation", "mAP50_val", "mAP50_95_val", "precision_val", "recall_val"]
print(df_results[cols_display].to_string(index=False, float_format="{:.4f}".format))

df_results.to_csv("experiments_results.csv", index=False)
print("\nTableau sauvegardé : experiments_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

runs_short = [r.split("_", 1)[1] for r in df_results["run"]]
colors_map = {"yolov8n.pt": "#4CAF50", "yolov8s.pt": "#2196F3"}
bar_colors = [colors_map[m] for m in df_results["model"]]

bars1 = axes[0].barh(runs_short, df_results["mAP50_val"], color=bar_colors, edgecolor="black")
axes[0].set_title("mAP@50 sur validation (↑ meilleur)", fontsize=13)
axes[0].set_xlabel("mAP@50"); axes[0].set_xlim(0, 1)
for bar, val in zip(bars1, df_results["mAP50_val"]):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontsize=9)

bars2 = axes[1].barh(runs_short, df_results["mAP50_95_val"], color=bar_colors, edgecolor="black")
axes[1].set_title("mAP@50-95 sur validation (↑ meilleur)", fontsize=13)
axes[1].set_xlabel("mAP@50-95"); axes[1].set_xlim(0, 1)
for bar, val in zip(bars2, df_results["mAP50_95_val"]):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=m.replace(".pt","")) for m, c in colors_map.items()]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=10, title="Modèle")

plt.suptitle("Comparaison des expériences — Validation", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig("experiments_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_perf = df_results.groupby("model")["mAP50_val"].mean().sort_values(ascending=False)
axes[0].bar(model_perf.index.str.replace(".pt",""), model_perf.values,
            color=["#4CAF50","#2196F3","#FF5722"][:len(model_perf)], edgecolor="black")
axes[0].set_title("mAP@50 moyen par architecture", fontsize=12)
axes[0].set_ylabel("mAP@50 moyen"); axes[0].set_ylim(0, 1)
for i, v in enumerate(model_perf.values):
    axes[0].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10)

lr_perf = df_results.groupby("lr0")["mAP50_val"].mean()
axes[1].bar([str(lr) for lr in lr_perf.index], lr_perf.values,
            color=["#9C27B0","#FF9800"], edgecolor="black")
axes[1].set_title("mAP@50 moyen par learning rate", fontsize=12)
axes[1].set_ylabel("mAP@50 moyen"); axes[1].set_ylim(0, 1)
for i, v in enumerate(lr_perf.values):
    axes[1].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10)

aug_perf = df_results.groupby("augmentation")["mAP50_val"].mean()
axes[2].bar(aug_perf.index, aug_perf.values,
            color=["#607D8B","#F44336"], edgecolor="black")
axes[2].set_title("mAP@50 moyen par augmentation", fontsize=12)
axes[2].set_ylabel("mAP@50 moyen"); axes[2].set_ylim(0, 1)
for i, v in enumerate(aug_perf.values):
    axes[2].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10)

plt.suptitle("Analyse de l'impact des facteurs expérimentaux", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("factor_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Sélection & Sauvegarde du Meilleur Modèle

In [ ]:
best_run = df_results.iloc[0]
print("="*60)
print("MEILLEUR MODÈLE SÉLECTIONNÉ")
print("="*60)
print(f"  Run          : {best_run['run']}")
print(f"  Architecture : {best_run['model']}")
print(f"  LR initial   : {best_run['lr0']}")
print(f"  Optimizer    : {best_run['optimizer']}")
print(f"  Augmentation : {best_run['augmentation']}")
print(f"  mAP@50 val   : {best_run['mAP50_val']:.4f}")
print(f"  mAP@50-95 val: {best_run['mAP50_95_val']:.4f}")

BEST_MODEL_PATH = Path(best_run["best_pt"])

dest = MODELS_DIR / "best_simpsons_model.pt"
shutil.copy(str(BEST_MODEL_PATH), str(dest))
print(f"\n✅ Modèle sauvegardé sur Drive : {dest}")

shutil.copy("experiments_results.csv", str(MODELS_DIR / "experiments_results.csv"))
print(f"✅ Tableau de résultats sauvegardé")

In [ ]:
best_run_dir = Path("/content/runs/detect") / PROJECT / best_run["run"]
if not best_run_dir.exists():
    best_run_dir = Path("/content/runs/detect") / best_run["run"]

print(f"Dossier du meilleur run : {best_run_dir}")

# ── Courbes générées automatiquement par YOLOv8 ──────────────────────
curves = {
    "results.png"                      : "Courbes loss & mAP",
    "confusion_matrix_normalized.png"  : "Matrice de confusion normalisée",
    "confusion_matrix.png"             : "Matrice de confusion",
}

for fname, title in curves.items():
    fpath = best_run_dir / fname
    if fpath.exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(fpath))
        plt.title(f"{title} — {best_run['run']}", fontsize=13)
        plt.axis("off"); plt.tight_layout(); plt.show()
    else:
        print(f"[!] Non trouvé : {fpath}")

# ── Courbe Précision-Rappel générée manuellement ─────────────────────
best_model_for_curves = YOLO(str(BEST_MODEL_PATH))
metrics_for_curves = best_model_for_curves.val(
    data    = DATA_YAML_STR,
    split   = "val",
    imgsz   = IMG_SIZE,
    device  = DEVICE,
    verbose = False,
)

# Récupération des valeurs P/R par classe
precision_vals = metrics_for_curves.box.p   # précision par seuil
recall_vals    = metrics_for_curves.box.r    # rappel par seuil

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe P-R globale (moyenne des classes)
try:
    px = metrics_for_curves.box.px   # seuils de confiance
    py = metrics_for_curves.box.py   # précision par classe et seuil
    for i, name in enumerate(CLASS_NAMES):
        axes[0].plot(metrics_for_curves.box.curves_results[0],
                     metrics_for_curves.box.curves_results[1][i],
                     label=name, linewidth=1.5)
    axes[0].set_xlabel("Rappel"); axes[0].set_ylabel("Précision")
    axes[0].set_title("Courbe Précision-Rappel par classe")
    axes[0].legend(); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.05)
    axes[0].grid(alpha=0.3)
except Exception:
    # Fallback : barplot précision/rappel par classe
    x = range(len(CLASS_NAMES))
    axes[0].bar([n for n in CLASS_NAMES], precision_vals,
                color=sns.color_palette("Set2", len(CLASS_NAMES)), edgecolor="black")
    axes[0].set_title("Précision par classe (val)"); axes[0].set_ylim(0, 1.05)
    for i, v in enumerate(precision_vals):
        axes[0].text(i, v+0.01, f"{v:.2f}", ha="center", fontsize=9)

# F1 par classe
f1_vals = 2 * precision_vals * recall_vals / (precision_vals + recall_vals + 1e-9)
axes[1].bar(CLASS_NAMES, f1_vals,
            color=sns.color_palette("husl", len(CLASS_NAMES)), edgecolor="black")
axes[1].axhline(y=f1_vals.mean(), color="red", linestyle="--",
                label=f"F1 moyen = {f1_vals.mean():.3f}")
axes[1].set_title("F1-score par classe (val)"); axes[1].set_ylim(0, 1.05)
axes[1].legend()
for i, v in enumerate(f1_vals):
    axes[1].text(i, v+0.01, f"{v:.2f}", ha="center", fontsize=9)

plt.suptitle(f"Métriques — {best_run['run']}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pr_f1_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : pr_f1_curves.png")

## 9. Conclusion

Ce notebook a conduit **4 expériences** couvrant :
- 2 architectures YOLOv8 (nano, small)
- 2 learning rates (1e-3 et 5e-4)
- 2 stratégies d'augmentation (standard et renforcée)

Le meilleur modèle a été sélectionné sur le **mAP@50 du set de validation** et sauvegardé sur Google Drive sous le nom `best_simpsons_model.pt`.

➡️ **L'évaluation finale sur le jeu de test est réalisée dans le notebook `evaluate_simpson.ipynb`.**